# Risk-Sensitive Reinforcement Learning for Trading
## Comparison: PPO vs CVaR-PPO

Author: Student Project  
Date: February 2026  
Environment: Kaggle GPU

---

## Project Overview
Implement and compare two reinforcement learning methods for algorithmic trading:
1. PPO (Risk-Neutral Baseline)
2. CVaR-PPO (Risk-Sensitive)

Against baseline: Buy and Hold strategy

---

## Table of Contents
1. Setup and Configuration
2. Data Loading with Quality Checks
3. Feature Engineering
4. Trading Environment
5. Method 1: PPO
6. Method 2: CVaR-PPO
7. Baseline Strategy
8. Evaluation and Comparison
9. Visualization
10. Conclusion

## 1. Setup and Configuration

In [ ]:
# Install packages (Kaggle compatible)
import sys
import subprocess

def install_package(package):
    try:
        __import__(package.split('[')[0])
        print(f"Package {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"{package} installed successfully")

# Install required packages
packages = ['yfinance', 'ta', 'gym', 'torch', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly']
for pkg in packages:
    install_package(pkg)

print("\nAll packages installed!")

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import yfinance as yf
import ta
import gym
from gym import spaces

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Normal

from collections import deque
import pickle
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

print("All packages imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Configuration
CONFIG = {
    # Data parameters
    'SYMBOL': 'SPY',  # S&P 500 ETF - most liquid, reliable data
    'START_DATE': '2015-01-01',  # Extended to 10 years for better training
    'END_DATE': '2024-12-31',
    'INTERVAL': '1d',
    
    # Data split ratios
    'TRAIN_RATIO': 0.7,
    'VAL_RATIO': 0.15,
    'TEST_RATIO': 0.15,
    
    # Environment parameters
    'INITIAL_BALANCE': 10000,
    'TRANSACTION_COST': 0.001,  # 0.1% per trade
    'SLIPPAGE': 0.0005,  # 0.05%
    
    # PPO parameters (reverted to original - overfitting was detected)
    'PPO_LEARNING_RATE': 3e-4,  # Back to original
    'PPO_GAMMA': 0.99,
    'PPO_EPSILON': 0.2,
    'PPO_EPOCHS': 10,
    'PPO_BATCH_SIZE': 64,
    'PPO_HIDDEN_DIM': 256,
    
    # CVaR-PPO parameters (IMPROVED v4 - balanced approach)
    'CVAR_ALPHA': 0.15,  # Focus on worst 15% returns
    'CVAR_LAMBDA': 0.16,  # Initial weight for CVaR penalty (moderate)
    'CVAR_LAMBDA_DECAY': 0.996,  # Moderate decay (balance between v1 and v3)
    'CVAR_LAMBDA_MIN': 0.10,  # Moderate minimum (allows gradual risk-taking)
    'CVAR_LEARNING_RATE': 1.5e-4,  # Moderate LR (fast enough to converge)
    'CVAR_CLIP_RATIO': 0.22,  # Moderate clipping (balance between v1 and v3)
    'CVAR_L2_LAMBDA': 1e-5,  # Very light L2 regularization
    
    # Training parameters
    'NUM_EPISODES': 100,
    'CVAR_NUM_EPISODES': 100,  # Balanced: enough to learn, not too much to overfit
    'UPDATE_INTERVAL': 512,
    
    # Device
    'DEVICE': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print("Configuration loaded:")
print(f"  Symbol: {CONFIG['SYMBOL']}")
print(f"  Period: {CONFIG['START_DATE']} to {CONFIG['END_DATE']}")
print(f"  Device: {CONFIG['DEVICE']}")
print(f"  Training episodes: PPO={CONFIG['NUM_EPISODES']}, CVaR-PPO={CONFIG['CVAR_NUM_EPISODES']}")


## 2. Data Loading with Quality Checks

In [ ]:
def download_and_validate_data(symbol, start_date, end_date, interval='1d'):
    """
    Download stock data from Yahoo Finance with comprehensive quality checks
    
    Parameters:
        symbol: Stock ticker symbol
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
        interval: Data interval (1d, 1h, etc.)
    
    Returns:
        DataFrame with validated OHLCV data
    """
    print(f"\n{'='*60}")
    print(f"DOWNLOADING DATA: {symbol}")
    print(f"{'='*60}")
    print(f"Period: {start_date} to {end_date}")
    print(f"Interval: {interval}")
    
    try:
        # Download data
        ticker = yf.Ticker(symbol)
        df = ticker.history(start=start_date, end=end_date, interval=interval)
        
        if df.empty:
            raise ValueError(f"No data downloaded for {symbol}")
        
        print(f"\nInitial data: {len(df)} rows")
        
        # Reset index and clean column names
        df.reset_index(inplace=True)
        df.columns = df.columns.str.lower()
        
        # Keep only necessary columns
        required_cols = ['date', 'open', 'high', 'low', 'close', 'volume']
        df = df[required_cols]
        
        # Quality checks
        print(f"\n{'='*60}")
        print("DATA QUALITY CHECKS")
        print(f"{'='*60}")
        
        # 1. Check for missing values
        missing = df.isnull().sum()
        print(f"\n1. Missing values:")
        for col, count in missing.items():
            if count > 0:
                print(f"   {col}: {count} ({count/len(df)*100:.2f}%)")
        if missing.sum() == 0:
            print("   No missing values")
        
        # 2. Check for duplicate dates
        duplicates = df['date'].duplicated().sum()
        print(f"\n2. Duplicate dates: {duplicates}")
        if duplicates > 0:
            print(f"   Removing {duplicates} duplicates...")
            df = df[~df['date'].duplicated(keep='first')]
        
        # 3. Check for zero/negative prices
        price_cols = ['open', 'high', 'low', 'close']
        zero_prices = (df[price_cols] <= 0).sum().sum()
        print(f"\n3. Zero/negative prices: {zero_prices}")
        if zero_prices > 0:
            print("   Removing rows with invalid prices...")
            df = df[(df[price_cols] > 0).all(axis=1)]
        
        # 4. Check for zero volume
        zero_volume = (df['volume'] == 0).sum()
        print(f"\n4. Zero volume days: {zero_volume} ({zero_volume/len(df)*100:.2f}%)")
        
        # 5. Check price consistency (High >= Low)
        inconsistent = (df['high'] < df['low']).sum()
        print(f"\n5. Inconsistent OHLC: {inconsistent}")
        if inconsistent > 0:
            print("   Fixing inconsistent prices...")
            mask = df['high'] < df['low']
            df.loc[mask, ['high', 'low']] = df.loc[mask, ['low', 'high']].values
        
        # 6. Check for extreme price jumps (>50% in one day)
        df['price_change'] = df['close'].pct_change().abs()
        extreme_jumps = (df['price_change'] > 0.5).sum()
        print(f"\n6. Extreme price jumps (>50%): {extreme_jumps}")
        if extreme_jumps > 0:
            jump_dates = df[df['price_change'] > 0.5]['date'].tolist()
            print(f"   Dates: {jump_dates[:3]}...")  # Show first 3
        df.drop('price_change', axis=1, inplace=True)
        
        # 7. Remove NaN values
        df.dropna(inplace=True)
        
        # 8. Sort by date
        df.sort_values('date', inplace=True)
        df.reset_index(drop=True, inplace=True)
        
        # Final stats
        print(f"\n{'='*60}")
        print("FINAL DATA SUMMARY")
        print(f"{'='*60}")
        print(f"Total rows: {len(df)}")
        print(f"Date range: {df['date'].min()} to {df['date'].max()}")
        print(f"Trading days: {len(df)}")
        print(f"\nPrice statistics:")
        print(f"  Min close: ${df['close'].min():.2f}")
        print(f"  Max close: ${df['close'].max():.2f}")
        print(f"  Mean close: ${df['close'].mean():.2f}")
        print(f"  Std close: ${df['close'].std():.2f}")
        
        # Check if we have enough data
        min_required_rows = 252 * 2  # At least 2 years of daily data
        if len(df) < min_required_rows:
            print(f"\nWARNING: Only {len(df)} rows, recommended minimum is {min_required_rows}")
        else:
            print(f"\nData quality: GOOD ({len(df)} rows available)")
        
        return df
        
    except Exception as e:
        print(f"\nERROR downloading data: {str(e)}")
        raise

# Download and validate data
raw_data = download_and_validate_data(
    symbol=CONFIG['SYMBOL'],
    start_date=CONFIG['START_DATE'],
    end_date=CONFIG['END_DATE'],
    interval=CONFIG['INTERVAL']
)

# Display sample
print(f"\n{'='*60}")
print("DATA SAMPLE")
print(f"{'='*60}")
print(raw_data.head(10))

## 3. Feature Engineering

In [ ]:
def add_technical_indicators(df):
    """
    Add technical indicators with error handling
    
    Indicators:
    - SMA (Simple Moving Average): 10, 20, 50 days
    - RSI (Relative Strength Index): 14 days
    - MACD (Moving Average Convergence Divergence)
    - Bollinger Bands
    - ATR (Average True Range)
    - Returns
    """
    print(f"\n{'='*60}")
    print("ADDING TECHNICAL INDICATORS")
    print(f"{'='*60}")
    
    df = df.copy()
    initial_rows = len(df)
    
    try:
        # Simple Moving Averages
        print("Adding SMA indicators...")
        df['sma_10'] = ta.trend.sma_indicator(df['close'], window=10)
        df['sma_20'] = ta.trend.sma_indicator(df['close'], window=20)
        df['sma_50'] = ta.trend.sma_indicator(df['close'], window=50)
        
        # RSI
        print("Adding RSI...")
        df['rsi'] = ta.momentum.rsi(df['close'], window=14)
        
        # MACD
        print("Adding MACD...")
        macd = ta.trend.MACD(df['close'])
        df['macd'] = macd.macd()
        df['macd_signal'] = macd.macd_signal()
        df['macd_diff'] = macd.macd_diff()
        
        # Bollinger Bands
        print("Adding Bollinger Bands...")
        bollinger = ta.volatility.BollingerBands(df['close'])
        df['bb_high'] = bollinger.bollinger_hband()
        df['bb_low'] = bollinger.bollinger_lband()
        df['bb_mid'] = bollinger.bollinger_mavg()
        
        # ATR
        print("Adding ATR...")
        df['atr'] = ta.volatility.average_true_range(df['high'], df['low'], df['close'])
        
        # Price-based features
        print("Adding return features...")
        df['returns'] = df['close'].pct_change()
        df['log_returns'] = np.log(df['close'] / df['close'].shift(1))
        
        # Volume features
        print("Adding volume features...")
        df['volume_sma'] = ta.trend.sma_indicator(df['volume'], window=20)
        
        # Remove NaN values created by indicators
        df.dropna(inplace=True)
        
        rows_lost = initial_rows - len(df)
        print(f"\nFeature engineering complete!")
        print(f"  Total features: {len(df.columns)}")
        print(f"  Rows removed (NaN): {rows_lost} ({rows_lost/initial_rows*100:.2f}%)")
        print(f"  Final rows: {len(df)}")
        
        # Verify no inf or nan values
        inf_count = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
        nan_count = df.isnull().sum().sum()
        print(f"  Inf values: {inf_count}")
        print(f"  NaN values: {nan_count}")
        
        if inf_count > 0 or nan_count > 0:
            print("  WARNING: Found inf or nan values, cleaning...")
            df.replace([np.inf, -np.inf], np.nan, inplace=True)
            df.dropna(inplace=True)
            print(f"  Cleaned. Final rows: {len(df)}")
        
        return df
        
    except Exception as e:
        print(f"\nERROR in feature engineering: {str(e)}")
        raise

# Add technical indicators
data_with_features = add_technical_indicators(raw_data)

# Display features
print(f"\n{'='*60}")
print("FEATURES")
print(f"{'='*60}")
print(f"Features ({len(data_with_features.columns)}):")
for i, col in enumerate(data_with_features.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
def split_data(df, train_ratio, val_ratio, test_ratio):
    """
    Split data chronologically into train, validation, and test sets
    """
    print(f"\n{'='*60}")
    print("SPLITTING DATA")
    print(f"{'='*60}")
    
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1"
    
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    
    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()
    
    print(f"Train set: {len(train_df)} samples ({100*train_ratio:.0f}%)")
    print(f"  Period: {train_df['date'].min()} to {train_df['date'].max()}")
    print(f"  Trading days: {len(train_df)}")
    
    print(f"\nValidation set: {len(val_df)} samples ({100*val_ratio:.0f}%)")
    print(f"  Period: {val_df['date'].min()} to {val_df['date'].max()}")
    print(f"  Trading days: {len(val_df)}")
    
    print(f"\nTest set: {len(test_df)} samples ({100*test_ratio:.0f}%)")
    print(f"  Period: {test_df['date'].min()} to {test_df['date'].max()}")
    print(f"  Trading days: {len(test_df)}")
    
    return train_df, val_df, test_df

# Split data
train_data, val_data, test_data = split_data(
    data_with_features,
    CONFIG['TRAIN_RATIO'],
    CONFIG['VAL_RATIO'],
    CONFIG['TEST_RATIO']
)

## 4. Trading Environment

In [ ]:
class TradingEnvironment(gym.Env):
    """
    Custom Trading Environment for Reinforcement Learning
    
    State space: Technical indicators + portfolio information
    Action space: Continuous [-1, 1]
        -1 = Sell all
         0 = Hold
        +1 = Buy all
    
    Reward: Portfolio return percentage with transaction costs
    """
    
    def __init__(self, df, initial_balance=10000, transaction_cost=0.001, slippage=0.0005):
        super(TradingEnvironment, self).__init__()
        
        self.df = df.reset_index(drop=True)
        self.initial_balance = initial_balance
        self.transaction_cost = transaction_cost
        self.slippage = slippage
        
        # Features for state (exclude date and basic OHLC)
        self.feature_columns = [col for col in df.columns 
                                if col not in ['date', 'open', 'high', 'low']]
        
        # Normalize market features (critical for neural network training!)
        self.feature_means = df[self.feature_columns].mean()
        self.feature_stds = df[self.feature_columns].std() + 1e-8
        
        # Action space: continuous [-1, 1]
        self.action_space = spaces.Box(low=-1, high=1, shape=(1,), dtype=np.float32)
        
        # Observation space: features + portfolio info
        n_features = len(self.feature_columns) + 3  # +3 for balance, shares, portfolio_value
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(n_features,), dtype=np.float32
        )
        
        # Episode variables
        self.current_step = 0
        self.balance = initial_balance
        self.shares = 0
        self.portfolio_value = initial_balance
        self.history = []
        
    def reset(self):
        """Reset environment to initial state"""
        self.current_step = 0
        self.balance = self.initial_balance
        self.shares = 0
        self.portfolio_value = self.initial_balance
        self.history = []
        
        return self._get_observation()
    
    def _get_observation(self):
        """Get current state observation"""
        # Market features (NORMALIZED!)
        market_features_raw = self.df.loc[self.current_step, self.feature_columns].values
        market_features = (market_features_raw - self.feature_means.values) / self.feature_stds.values
        
        # Portfolio features (already normalized)
        current_price = self.df.loc[self.current_step, 'close']
        portfolio_features = np.array([
            self.balance / self.initial_balance,
            self.shares * current_price / self.initial_balance,
            self.portfolio_value / self.initial_balance
        ])
        
        observation = np.concatenate([market_features, portfolio_features]).astype(np.float32)
        return observation
    
    def step(self, action):
        """Execute one trading step"""
        current_price = self.df.loc[self.current_step, 'close']
        action = float(action[0])  # Convert to scalar
        
        # Calculate portfolio value before action
        old_portfolio_value = self.balance + self.shares * current_price
        
        # Execute trading action
        if action > 0.01:  # Buy (small threshold to avoid noise)
            amount_to_invest = self.balance * abs(action)
            if amount_to_invest > 0:  # Only trade if we have balance
                execution_price = current_price * (1 + self.slippage)
                shares_to_buy = (amount_to_invest * (1 - self.transaction_cost)) / execution_price
                
                self.shares += shares_to_buy
                self.balance -= amount_to_invest
            
        elif action < -0.01:  # Sell (small threshold to avoid noise)
            shares_to_sell = self.shares * abs(action)
            if shares_to_sell > 0:  # Only sell if we have shares
                execution_price = current_price * (1 - self.slippage)
                proceeds = shares_to_sell * execution_price * (1 - self.transaction_cost)
                
                self.shares -= shares_to_sell
                self.balance += proceeds
        
        # Move to next step
        self.current_step += 1
        
        # Calculate new portfolio value
        if self.current_step < len(self.df):
            new_price = self.df.loc[self.current_step, 'close']
            new_portfolio_value = self.balance + self.shares * new_price
        else:
            new_portfolio_value = self.balance + self.shares * current_price
        
        # Calculate reward (percentage return)
        reward = (new_portfolio_value - old_portfolio_value) / (old_portfolio_value + 1e-10)
        
        # Update portfolio value
        self.portfolio_value = new_portfolio_value
        
        # Check if episode is done
        done = self.current_step >= len(self.df) - 1
        
        # Store history
        self.history.append({
            'step': self.current_step,
            'action': action,
            'balance': self.balance,
            'shares': self.shares,
            'price': current_price,
            'portfolio_value': self.portfolio_value,
            'reward': reward
        })
        
        # Get next observation
        observation = self._get_observation()
        info = {}
        
        return observation, reward, done, info

print("Trading Environment defined")

# Test environment
print("\nTesting environment...")
test_env = TradingEnvironment(
    train_data,
    initial_balance=CONFIG['INITIAL_BALANCE'],
    transaction_cost=CONFIG['TRANSACTION_COST'],
    slippage=CONFIG['SLIPPAGE']
)

obs = test_env.reset()
print(f"  Observation shape: {obs.shape}")
print(f"  Action space: {test_env.action_space}")
print(f"  First 5 market features (normalized): {obs[:5]}")
print(f"  Last 3 portfolio features: {obs[-3:]}")
print(f"  Feature mean: {obs.mean():.4f}, std: {obs.std():.4f}")
print("Environment test passed!")

## 5. Method 1: PPO (Risk-Neutral Baseline)

In [ ]:
class ActorCritic(nn.Module):
    """Actor-Critic Network for PPO"""
    
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super(ActorCritic, self).__init__()
        
        # Shared layers
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Actor head (policy)
        self.actor_mean = nn.Linear(hidden_dim, action_dim)
        self.actor_log_std = nn.Parameter(torch.zeros(action_dim))  # Reverted - original was better
        
        # Critic head (value function)
        self.critic = nn.Linear(hidden_dim, 1)
        
    def forward(self, state):
        return self.shared(state)
    
    def act(self, state):
        shared_features = self.forward(state)
        action_mean = self.actor_mean(shared_features)
        action_std = torch.exp(self.actor_log_std)
        
        dist = Normal(action_mean, action_std)
        action = dist.sample()
        action_log_prob = dist.log_prob(action).sum(dim=-1)
        
        # Clamp action to [-1, 1]
        action = torch.tanh(action)
        
        return action, action_log_prob
    
    def evaluate(self, state, action):
        shared_features = self.forward(state)
        
        action_mean = self.actor_mean(shared_features)
        action_std = torch.exp(self.actor_log_std)
        
        dist = Normal(action_mean, action_std)
        
        # Inverse tanh for log prob calculation
        action_untanh = torch.atanh(torch.clamp(action, -0.999, 0.999))
        action_log_prob = dist.log_prob(action_untanh).sum(dim=-1)
        
        # Adjust for tanh squashing
        action_log_prob -= torch.log(1 - action**2 + 1e-6).sum(dim=-1)
        
        dist_entropy = dist.entropy().sum(dim=-1)
        state_value = self.critic(shared_features)
        
        return action_log_prob, state_value, dist_entropy

print("Actor-Critic network defined")

In [ ]:
class PPOAgent:
    """Proximal Policy Optimization Agent"""
    
    def __init__(self, env, config):
        self.env = env
        self.config = config
        self.device = config['DEVICE']
        
        state_dim = env.observation_space.shape[0]
        action_dim = env.action_space.shape[0]
        
        self.policy = ActorCritic(
            state_dim, 
            action_dim, 
            hidden_dim=config['PPO_HIDDEN_DIM']
        ).to(self.device)
        
        self.optimizer = optim.Adam(
            self.policy.parameters(), 
            lr=config['PPO_LEARNING_RATE']
        )
        
        self.gamma = config['PPO_GAMMA']
        self.epsilon = config['PPO_EPSILON']
        self.epochs = config['PPO_EPOCHS']
        self.batch_size = config['PPO_BATCH_SIZE']
        
        # Storage
        self.memory = {
            'states': [],
            'actions': [],
            'rewards': [],
            'log_probs': [],
            'values': [],
            'dones': []
        }
        
        # Training history
        self.training_history = {
            'episode_rewards': [],
            'episode_lengths': [],
            'policy_losses': [],
            'value_losses': [],
            'portfolio_values': []
        }
    
    def select_action(self, state):
        """Select action using current policy"""
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            action, action_log_prob = self.policy.act(state)
        
        return action.cpu().numpy()[0], action_log_prob.cpu().item()
    
    def store_transition(self, state, action, reward, log_prob, value, done):
        """Store transition in memory"""
        self.memory['states'].append(state)
        self.memory['actions'].append(action)
        self.memory['rewards'].append(reward)
        self.memory['log_probs'].append(log_prob)
        self.memory['values'].append(value)
        self.memory['dones'].append(done)
    
    def compute_returns(self):
        """Compute discounted returns"""
        returns = []
        R = 0
        
        for i in reversed(range(len(self.memory['rewards']))):
            if self.memory['dones'][i]:
                R = 0
            R = self.memory['rewards'][i] + self.gamma * R
            returns.insert(0, R)
        
        returns = torch.tensor(returns, dtype=torch.float32).to(self.device)
        values = torch.tensor(self.memory['values'], dtype=torch.float32).to(self.device)
        
        advantages = returns - values
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        return returns, advantages
    
    def update(self):
        """Update policy using PPO"""
        returns, advantages = self.compute_returns()
        
        states = torch.FloatTensor(np.array(self.memory['states'])).to(self.device)
        actions = torch.FloatTensor(np.array(self.memory['actions'])).to(self.device)
        old_log_probs = torch.FloatTensor(self.memory['log_probs']).to(self.device)
        
        policy_losses = []
        value_losses = []
        
        for _ in range(self.epochs):
            log_probs, state_values, dist_entropy = self.policy.evaluate(states, actions)
            state_values = state_values.squeeze()
            
            # Compute ratio and clipped surrogate
            ratio = torch.exp(log_probs - old_log_probs)
            surr1 = ratio * advantages
            surr2 = torch.clamp(ratio, 1 - self.epsilon, 1 + self.epsilon) * advantages
            
            # Policy loss
            policy_loss = -torch.min(surr1, surr2).mean()
            
            # Value loss
            value_loss = F.mse_loss(state_values, returns)
            
            # Total loss
            loss = policy_loss + 0.5 * value_loss - 0.01 * dist_entropy.mean()
            
            # Optimize
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
            self.optimizer.step()
            
            policy_losses.append(policy_loss.item())
            value_losses.append(value_loss.item())
        
        # Clear memory
        self.memory = {
            'states': [], 'actions': [], 'rewards': [],
            'log_probs': [], 'values': [], 'dones': []
        }
        
        return np.mean(policy_losses), np.mean(value_losses)
    
    def train(self, num_episodes=100, update_interval=512):
        """Train the agent"""
        print(f"\nTraining PPO for {num_episodes} episodes...")
        print(f"{'='*60}")
        total_steps = 0
        
        for episode in range(num_episodes):
            state = self.env.reset()
            episode_reward = 0
            episode_length = 0
            done = False
            
            while not done:
                action, log_prob = self.select_action(state)
                
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
                with torch.no_grad():
                    value = self.policy.critic(self.policy.forward(state_tensor)).item()
                
                next_state, reward, done, _ = self.env.step(action)
                
                self.store_transition(state, action, reward, log_prob, value, done)
                
                state = next_state
                episode_reward += reward
                episode_length += 1
                total_steps += 1
                
                if total_steps % update_interval == 0:
                    policy_loss, value_loss = self.update()
                    self.training_history['policy_losses'].append(policy_loss)
                    self.training_history['value_losses'].append(value_loss)
            
            final_portfolio_value = self.env.portfolio_value
            
            self.training_history['episode_rewards'].append(episode_reward)
            self.training_history['episode_lengths'].append(episode_length)
            self.training_history['portfolio_values'].append(final_portfolio_value)
            
            if (episode + 1) % 10 == 0:
                avg_reward = np.mean(self.training_history['episode_rewards'][-10:])
                avg_portfolio = np.mean(self.training_history['portfolio_values'][-10:])
                progress = (episode + 1) / num_episodes * 100
                print(f"[{progress:5.1f}%] Episode {episode + 1}/{num_episodes}: "
                      f"Avg Reward: {avg_reward:+.4f}, "
                      f"Portfolio: ${avg_portfolio:,.2f}")
        
        print(f"{'='*60}")
        print("PPO training completed")
        return self.training_history
    
    def evaluate(self, env):
        """Evaluate trained policy"""
        state = env.reset()
        done = False
        
        while not done:
            action, _ = self.select_action(state)
            state, reward, done, _ = env.step(action)
        
        return env.history, env.portfolio_value

print("PPO Agent defined")

In [ ]:
# Train PPO Agent
print(f"\n{'='*60}")
print("TRAINING PPO (Risk-Neutral Baseline)")
print(f"{'='*60}")

train_env_ppo = TradingEnvironment(
    train_data,
    initial_balance=CONFIG['INITIAL_BALANCE'],
    transaction_cost=CONFIG['TRANSACTION_COST'],
    slippage=CONFIG['SLIPPAGE']
)

ppo_agent = PPOAgent(train_env_ppo, CONFIG)
ppo_training_history = ppo_agent.train(
    num_episodes=CONFIG['NUM_EPISODES'], 
    update_interval=CONFIG['UPDATE_INTERVAL']
)

print(f"\nPPO Training Complete!")
print(f"  Final Portfolio Value: ${ppo_training_history['portfolio_values'][-1]:.2f}")
print(f"  Total Episodes: {len(ppo_training_history['episode_rewards'])}")

In [ ]:
# Save PPO Model
print(f"\n{'='*60}")
print("SAVING PPO MODEL")
print(f"{'='*60}")

ppo_model_path = 'ppo_model.pth'
torch.save({
    'model_state_dict': ppo_agent.policy.state_dict(),
    'optimizer_state_dict': ppo_agent.optimizer.state_dict(),
    'training_history': ppo_training_history,
    'config': CONFIG
}, ppo_model_path)

print(f"PPO model saved to: {ppo_model_path}")
print(f"  Model parameters: {sum(p.numel() for p in ppo_agent.policy.parameters())}")
print(f"  Training episodes: {len(ppo_training_history['episode_rewards'])}")

In [ ]:
# Evaluate PPO on Validation Set  
print(f"\n{'='*60}")
print("EVALUATING PPO ON VALIDATION SET")
print(f"{'='*60}")

# Define calculate_metrics first (needed for validation)
def calculate_metrics(history, initial_balance=10000):
    """Calculate trading performance metrics"""
    portfolio_values = [h['portfolio_value'] for h in history]
    
    # Total return
    total_return = (portfolio_values[-1] - initial_balance) / initial_balance
    
    # Returns series
    returns = np.diff(portfolio_values) / (np.array(portfolio_values[:-1]) + 1e-10)
    
    # Sharpe Ratio (annualized)
    if len(returns) > 0 and np.std(returns) > 0:
        sharpe_ratio = np.mean(returns) / np.std(returns) * np.sqrt(252)
    else:
        sharpe_ratio = 0
    
    # Maximum Drawdown
    peak = np.maximum.accumulate(portfolio_values)
    drawdown = (portfolio_values - peak) / (peak + 1e-10)
    max_drawdown = np.min(drawdown)
    
    # Win rate
    if len(returns) > 0:
        win_rate = np.sum(returns > 0) / len(returns)
    else:
        win_rate = 0
    
    # Volatility (annualized)
    volatility = np.std(returns) * np.sqrt(252) if len(returns) > 0 else 0
    
    metrics = {
        'total_return': total_return,
        'final_value': portfolio_values[-1],
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'win_rate': win_rate,
        'volatility': volatility,
        'num_trades': len(history)
    }
    
    return metrics

# Evaluate PPO on validation
val_env_ppo = TradingEnvironment(
    val_data,
    initial_balance=CONFIG['INITIAL_BALANCE'],
    transaction_cost=CONFIG['TRANSACTION_COST'],
    slippage=CONFIG['SLIPPAGE']
)

ppo_val_history, ppo_val_value = ppo_agent.evaluate(val_env_ppo)
ppo_val_metrics = calculate_metrics(ppo_val_history, CONFIG['INITIAL_BALANCE'])

print("\nPPO Validation Results:")
for key, value in ppo_val_metrics.items():
    if key == 'final_value':
        print(f"  {key}: ${value:.2f}")
    elif key in ['total_return', 'sharpe_ratio', 'max_drawdown', 'win_rate', 'volatility']:
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Quick PPO Training Visualization
print(f"\n{'='*60}")
print("PPO TRAINING OVERVIEW")
print(f"{'='*60}")

# Simple 2-plot layout
fig = make_subplots(rows=1, cols=2, subplot_titles=('Learning Progress', 'Train vs Validation'))

# Plot 1: Portfolio evolution during training
episodes = list(range(1, len(ppo_training_history['portfolio_values']) + 1))
fig.add_trace(go.Scatter(x=episodes, y=ppo_training_history['portfolio_values'], 
                         name='Training Portfolio', line=dict(color='blue')), row=1, col=1)
fig.add_hline(y=CONFIG['INITIAL_BALANCE'], line_dash="dash", line_color="red", row=1, col=1)

# Plot 2: Final episode comparison
fig.add_trace(go.Scatter(y=[h['portfolio_value'] for h in ppo_agent.env.history],
                         name='Training (final)', line=dict(color='blue')), row=1, col=2)
fig.add_trace(go.Scatter(y=[h['portfolio_value'] for h in ppo_val_history],
                         name='Validation', line=dict(color='red')), row=1, col=2)

fig.update_xaxes(title_text="Episode", row=1, col=1)
fig.update_xaxes(title_text="Time Step", row=1, col=2)
fig.update_yaxes(title_text="Portfolio Value ($)", row=1, col=1)
fig.update_yaxes(title_text="Portfolio Value ($)", row=1, col=2)
fig.update_layout(title='PPO Training Analysis', template='plotly_white', width=1000, height=400)
fig.show()

# Stats Summary
train_return = (ppo_training_history['portfolio_values'][-1] - CONFIG['INITIAL_BALANCE']) / CONFIG['INITIAL_BALANCE'] * 100
print(f"\n📊 Training: {len(ppo_training_history['episode_rewards'])} episodes | Return: {train_return:+.2f}%")
print(f"📊 Validation: Return: {ppo_val_metrics['total_return']*100:+.2f}% | Sharpe: {ppo_val_metrics['sharpe_ratio']:.2f} | Drawdown: {ppo_val_metrics['max_drawdown']*100:.1f}%")

## 6. Method 2: CVaR-PPO (Risk-Sensitive)

In [ ]:
class CVaRPPOAgent(PPOAgent):
    """
    CVaR-PPO: Risk-Sensitive PPO with Conditional Value at Risk constraint
    
    IMPROVED VERSION v4:
    - Balanced approach between v1 (best test performance) và v3 (quá conservative)
    - 100 episodes: đủ để học, không quá nhiều để overfit
    - Lambda 0.16 → 0.10: kiểm soát risk nhưng vẫn cho phép model explore
    - Decay 0.996: vừa phải, lambda đạt min ở cuối training
    - L2 regularization nhẹ: chống overfit mà không làm model yếu đi
    """
    def __init__(self, env, config):
        # Initialize parent class but we'll override the optimizer
        super().__init__(env, config)
        self.cvar_alpha = config['CVAR_ALPHA']
        self.cvar_lambda_init = config['CVAR_LAMBDA']
        self.cvar_lambda = config['CVAR_LAMBDA']
        self.cvar_lambda_decay = config.get('CVAR_LAMBDA_DECAY', 0.996)
        self.cvar_lambda_min = config.get('CVAR_LAMBDA_MIN', 0.10)
        self.cvar_clip_ratio = config.get('CVAR_CLIP_RATIO', 0.22)
        self.l2_lambda = config.get('CVAR_L2_LAMBDA', 1e-5)
        # Re-create optimizer with L2 regularization (weight_decay)
        self.optimizer = optim.Adam(self.policy.parameters(), 
                                   lr=config['CVAR_LEARNING_RATE'], 
                                   weight_decay=self.l2_lambda)
        # Eta parameter for CVaR threshold (learnable)
        self.eta = nn.Parameter(torch.tensor(0.0, device=self.device))
        self.eta_optimizer = optim.Adam([self.eta], lr=config['CVAR_LEARNING_RATE'])
        # CVaR history
        self.training_history['cvar_losses'] = []
        self.training_history['eta_values'] = []
        self.training_history['lambda_values'] = []
    
    def compute_cvar_loss(self, returns):
        """Compute CVaR loss with improved stability"""
        sorted_returns, _ = torch.sort(returns)
        n_worst = max(1, int(len(sorted_returns) * self.cvar_alpha))
        worst_returns = sorted_returns[:n_worst]
        # CVaR is the average of worst alpha% returns
        cvar = worst_returns.mean()
        # Soft constraint: penalize returns below eta threshold
        cvar_violations = torch.relu(self.eta - sorted_returns)
        cvar_loss = cvar_violations.mean()
        # Add small regularization to prevent eta from becoming too negative
        eta_reg = 0.01 * torch.relu(-self.eta - 1.0)  # Penalize eta < -1
        return cvar_loss + eta_reg, cvar.item()
    
    def update(self):
        """
        Update policy with CVaR constraint  
        FIXED: Detach returns when computing eta loss to avoid double backward
        """
        returns, advantages = self.compute_returns()
        states = torch.FloatTensor(np.array(self.memory['states'])).to(self.device)
        actions = torch.FloatTensor(np.array(self.memory['actions'])).to(self.device)
        old_log_probs = torch.FloatTensor(self.memory['log_probs']).to(self.device)
        policy_losses = []
        value_losses = []
        cvar_losses = []
        for _ in range(self.epochs):
            # Evaluate actions
            log_probs, state_values, dist_entropy = self.policy.evaluate(states, actions)
            state_values = state_values.squeeze()
            # Compute ratio and clipped surrogate
            ratio = torch.exp(log_probs - old_log_probs)
            surr1 = ratio * advantages
            surr2 = torch.clamp(ratio, 1 - self.epsilon, 1 + self.epsilon) * advantages
            # Policy loss
            policy_loss = -torch.min(surr1, surr2).mean()
            # Value loss
            value_loss = F.mse_loss(state_values, returns)
            # CVaR loss (use detached returns for stability)
            cvar_loss, cvar_value = self.compute_cvar_loss(returns.detach())
            # Clip CVaR loss to prevent extreme gradients
            cvar_loss_clipped = torch.clamp(cvar_loss, 0, self.cvar_clip_ratio)
            # Total loss with adaptive CVaR penalty
            # Note: Keep entropy bonus higher (0.02 vs 0.01) for better exploration
            loss = policy_loss + 0.5 * value_loss - 0.02 * dist_entropy.mean() + self.cvar_lambda * cvar_loss_clipped
            # Optimize policy (L2 regularization applied via weight_decay in optimizer)
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
            self.optimizer.step()
            # Update eta (separate backward pass with detached returns)
            eta_returns = returns.detach()
            eta_cvar_loss, _ = self.compute_cvar_loss(eta_returns)
            eta_loss = -eta_cvar_loss
            self.eta_optimizer.zero_grad()
            eta_loss.backward()
            self.eta_optimizer.step()
            policy_losses.append(policy_loss.item())
            value_losses.append(value_loss.item())
            cvar_losses.append(cvar_loss.item())
        # Clear memory
        self.memory = {
            'states': [], 'actions': [], 'rewards': [],
            'log_probs': [], 'values': [], 'dones': []
        }
        # Decay lambda for adaptive risk-sensitivity (explore more early, exploit later)
        self.cvar_lambda = max(self.cvar_lambda_min, self.cvar_lambda * self.cvar_lambda_decay)
        self.training_history['cvar_losses'].append(np.mean(cvar_losses))
        self.training_history['eta_values'].append(self.eta.item())
        self.training_history['lambda_values'].append(self.cvar_lambda)
        return np.mean(policy_losses), np.mean(value_losses)
    
    def train(self, num_episodes=100, update_interval=512):
        """Train CVaR-PPO agent with improved hyperparameters (v4 - balanced approach)"""
        print(f"\nTraining IMPROVED CVaR-PPO v4 for {num_episodes} episodes...")
        print(f"  Alpha: {self.cvar_alpha} (worst {self.cvar_alpha*100:.0f}% returns)")
        print(f"  Lambda: {self.cvar_lambda_init} → {self.cvar_lambda_min} (decay={self.cvar_lambda_decay})")
        print(f"  Clip Ratio: {self.cvar_clip_ratio} | L2: {self.l2_lambda}")
        print(f"  Strategy: Balanced (v1 base + moderate anti-overfitting)")
        print(f"{'='*60}")
        total_steps = 0
        for episode in range(num_episodes):
            state = self.env.reset()
            episode_reward = 0
            episode_length = 0
            done = False
            while not done:
                action, log_prob = self.select_action(state)
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
                with torch.no_grad():
                    value = self.policy.critic(self.policy.forward(state_tensor)).item()
                next_state, reward, done, _ = self.env.step(action)
                self.store_transition(state, action, reward, log_prob, value, done)
                state = next_state
                episode_reward += reward
                episode_length += 1
                total_steps += 1
                if total_steps % update_interval == 0:
                    policy_loss, value_loss = self.update()
                    self.training_history['policy_losses'].append(policy_loss)
                    self.training_history['value_losses'].append(value_loss)
            final_portfolio_value = self.env.portfolio_value
            self.training_history['episode_rewards'].append(episode_reward)
            self.training_history['episode_lengths'].append(episode_length)
            self.training_history['portfolio_values'].append(final_portfolio_value)
            if (episode + 1) % 10 == 0:
                avg_reward = np.mean(self.training_history['episode_rewards'][-10:])
                avg_portfolio = np.mean(self.training_history['portfolio_values'][-10:])
                avg_cvar = np.mean(self.training_history['cvar_losses'][-10:]) if self.training_history['cvar_losses'] else 0
                current_lambda = self.training_history['lambda_values'][-1] if self.training_history['lambda_values'] else self.cvar_lambda
                progress = (episode + 1) / num_episodes * 100
                print(f"[{progress:5.1f}%] Episode {episode + 1}/{num_episodes}: "
                      f"Avg Reward: {avg_reward:+.4f}, "
                      f"Avg Portfolio: ${avg_portfolio:,.2f}, "
                      f"CVaR Loss: {avg_cvar:.4f}, λ: {current_lambda:.4f}")
        print(f"{'='*60}")
        print("CVaR-PPO training completed")
        return self.training_history

print("✓ IMPROVED CVaR-PPO Agent v4 defined")
print("  Strategy: Return to v1 foundation (best test: 11.60%) with moderate anti-overfitting")
print("  Key changes: 100 episodes, λ 0.16→0.10, decay 0.996, light L2 1e-5")


In [ ]:
# Train CVaR-PPO Agent
print(f"\n{'='*60}")
print("TRAINING CVaR-PPO (Risk-Sensitive)")
print(f"{'='*60}")

train_env_cvar = TradingEnvironment(
    train_data,
    initial_balance=CONFIG['INITIAL_BALANCE'],
    transaction_cost=CONFIG['TRANSACTION_COST'],
    slippage=CONFIG['SLIPPAGE']
)

cvar_ppo_agent = CVaRPPOAgent(train_env_cvar, CONFIG)
cvar_training_history = cvar_ppo_agent.train(
    num_episodes=CONFIG['CVAR_NUM_EPISODES'],  # Use dedicated CVaR episodes count
    update_interval=CONFIG['UPDATE_INTERVAL']
)

print(f"\nCVaR-PPO Training Complete!")
print(f"  Final Portfolio Value: ${cvar_training_history['portfolio_values'][-1]:.2f}")
print(f"  Total Episodes: {len(cvar_training_history['episode_rewards'])}")

In [ ]:
# Save CVaR-PPO Model
print(f"\n{'='*60}")
print("SAVING CVaR-PPO MODEL")
print(f"{'='*60}")

cvar_model_path = 'cvar_ppo_model.pth'
torch.save({
    'model_state_dict': cvar_ppo_agent.policy.state_dict(),
    'optimizer_state_dict': cvar_ppo_agent.optimizer.state_dict(),
    'eta_state_dict': cvar_ppo_agent.eta,
    'eta_optimizer_state_dict': cvar_ppo_agent.eta_optimizer.state_dict(),
    'training_history': cvar_training_history,
    'config': CONFIG
}, cvar_model_path)

print(f"CVaR-PPO model saved to: {cvar_model_path}")
print(f"  Model parameters: {sum(p.numel() for p in cvar_ppo_agent.policy.parameters())}")
print(f"  Training episodes: {len(cvar_training_history['episode_rewards'])}")


In [ ]:
# Evaluate CVaR-PPO on Validation Set  
print(f"\n{'='*60}")
print("EVALUATING CVaR-PPO ON VALIDATION SET")
print(f"{'='*60}")

# Evaluate CVaR-PPO on validation
val_env_cvar = TradingEnvironment(
    val_data,
    initial_balance=CONFIG['INITIAL_BALANCE'],
    transaction_cost=CONFIG['TRANSACTION_COST'],
    slippage=CONFIG['SLIPPAGE']
)

cvar_val_history, cvar_val_value = cvar_ppo_agent.evaluate(val_env_cvar)
cvar_val_metrics = calculate_metrics(cvar_val_history, CONFIG['INITIAL_BALANCE'])

print("\nCVaR-PPO Validation Results:")
for key, value in cvar_val_metrics.items():
    if key == 'final_value':
        print(f"  {key}: ${value:.2f}")
    elif key in ['total_return', 'sharpe_ratio', 'max_drawdown', 'win_rate', 'volatility']:
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Quick CVaR-PPO Training Visualization
print(f"\n{'='*60}")
print("CVaR-PPO TRAINING OVERVIEW")
print(f"{'='*60}")

# Simple 2-plot layout
fig = make_subplots(rows=1, cols=2, subplot_titles=('Learning Progress', 'Train vs Validation'))

# Plot 1: Portfolio evolution during training
episodes = list(range(1, len(cvar_training_history['portfolio_values']) + 1))
fig.add_trace(go.Scatter(x=episodes, y=cvar_training_history['portfolio_values'], 
                         name='Training Portfolio', line=dict(color='red')), row=1, col=1)
fig.add_hline(y=CONFIG['INITIAL_BALANCE'], line_dash="dash", line_color="gray", row=1, col=1)

# Plot 2: Final episode comparison
fig.add_trace(go.Scatter(y=[h['portfolio_value'] for h in cvar_ppo_agent.env.history],
                         name='Training (final)', line=dict(color='red')), row=1, col=2)
fig.add_trace(go.Scatter(y=[h['portfolio_value'] for h in cvar_val_history],
                         name='Validation', line=dict(color='orange')), row=1, col=2)

fig.update_xaxes(title_text="Episode", row=1, col=1)
fig.update_xaxes(title_text="Time Step", row=1, col=2)
fig.update_yaxes(title_text="Portfolio Value ($)", row=1, col=1)
fig.update_yaxes(title_text="Portfolio Value ($)", row=1, col=2)
fig.update_layout(title='CVaR-PPO Training Analysis', template='plotly_white', width=1000, height=400)
fig.show()

# Stats Summary
train_return = (cvar_training_history['portfolio_values'][-1] - CONFIG['INITIAL_BALANCE']) / CONFIG['INITIAL_BALANCE'] * 100
print(f"\n📊 Training: {len(cvar_training_history['episode_rewards'])} episodes | Return: {train_return:+.2f}%")
print(f"📊 Validation: Return: {cvar_val_metrics['total_return']*100:+.2f}% | Sharpe: {cvar_val_metrics['sharpe_ratio']:.2f} | Drawdown: {cvar_val_metrics['max_drawdown']*100:.1f}%")
print(f"📊 CVaR Loss: {cvar_training_history['cvar_losses'][-1]:.4f} | Eta: {cvar_training_history['eta_values'][-1]:.4f}")

## 7. Baseline: Buy and Hold Strategy

In [ ]:
def buy_and_hold_strategy(df, initial_balance=10000):
    """
    Buy and Hold baseline strategy
    
    Buy at first day, hold until last day
    """
    first_price = df['close'].iloc[0]
    last_price = df['close'].iloc[-1]
    
    shares = initial_balance / first_price
    final_value = shares * last_price
    
    portfolio_values = []
    for price in df['close']:
        portfolio_values.append(shares * price)
    
    history = []
    for i, (idx, row) in enumerate(df.iterrows()):
        history.append({
            'step': i,
            'action': 1.0 if i == 0 else 0.0,
            'balance': 0.0,
            'shares': shares,
            'price': row['close'],
            'portfolio_value': portfolio_values[i],
            'reward': 0.0
        })
    
    return history, final_value

print("Calculating Buy and Hold baseline...")

bh_train_history, bh_train_value = buy_and_hold_strategy(train_data, CONFIG['INITIAL_BALANCE'])
bh_val_history, bh_val_value = buy_and_hold_strategy(val_data, CONFIG['INITIAL_BALANCE'])
bh_test_history, bh_test_value = buy_and_hold_strategy(test_data, CONFIG['INITIAL_BALANCE'])

print(f"\nBuy and Hold Results:")
print(f"  Train set: ${bh_train_value:.2f}")
print(f"  Validation set: ${bh_val_value:.2f}")
print(f"  Test set: ${bh_test_value:.2f}")

## 8. Evaluation and Comparison

In [ ]:
print(f"\n{'='*60}")
print("EVALUATING ON TEST SET")
print(f"{'='*60}")

# Evaluate PPO on test set
test_env_ppo = TradingEnvironment(test_data, CONFIG['INITIAL_BALANCE'], 
                                  CONFIG['TRANSACTION_COST'], CONFIG['SLIPPAGE'])
ppo_test_history, ppo_test_value = ppo_agent.evaluate(test_env_ppo)
ppo_test_metrics = calculate_metrics(ppo_test_history, CONFIG['INITIAL_BALANCE'])

# Evaluate CVaR-PPO on test set
test_env_cvar = TradingEnvironment(test_data, CONFIG['INITIAL_BALANCE'], 
                                   CONFIG['TRANSACTION_COST'], CONFIG['SLIPPAGE'])
cvar_test_history, cvar_test_value = cvar_ppo_agent.evaluate(test_env_cvar)
cvar_test_metrics = calculate_metrics(cvar_test_history, CONFIG['INITIAL_BALANCE'])

# Buy and Hold metrics on test set
bh_test_metrics = calculate_metrics(bh_test_history, CONFIG['INITIAL_BALANCE'])

print("\nPPO (Risk-Neutral):")
for key, value in ppo_test_metrics.items():
    if key == 'final_value':
        print(f"  {key}: ${value:.2f}")
    elif key in ['total_return', 'sharpe_ratio', 'max_drawdown', 'win_rate', 'volatility']:
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

print("\nCVaR-PPO (Risk-Sensitive):")
for key, value in cvar_test_metrics.items():
    if key == 'final_value':
        print(f"  {key}: ${value:.2f}")
    elif key in ['total_return', 'sharpe_ratio', 'max_drawdown', 'win_rate', 'volatility']:
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

print("\nBuy and Hold:")
for key, value in bh_test_metrics.items():
    if key == 'final_value':
        print(f"  {key}: ${value:.2f}")
    elif key in ['total_return', 'sharpe_ratio', 'max_drawdown', 'win_rate', 'volatility']:
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame({
    'Method': ['PPO', 'CVaR-PPO', 'Buy & Hold'],
    'Final Value': [ppo_test_metrics['final_value'], cvar_test_metrics['final_value'], bh_test_metrics['final_value']],
    'Total Return': [ppo_test_metrics['total_return'], cvar_test_metrics['total_return'], bh_test_metrics['total_return']],
    'Sharpe Ratio': [ppo_test_metrics['sharpe_ratio'], cvar_test_metrics['sharpe_ratio'], bh_test_metrics['sharpe_ratio']],
    'Max Drawdown': [ppo_test_metrics['max_drawdown'], cvar_test_metrics['max_drawdown'], bh_test_metrics['max_drawdown']],
    'Win Rate': [ppo_test_metrics['win_rate'], cvar_test_metrics['win_rate'], bh_test_metrics['win_rate']],
    'Volatility': [ppo_test_metrics['volatility'], cvar_test_metrics['volatility'], bh_test_metrics['volatility']]
})

print(f"\n{'='*60}")
print("COMPARISON TABLE")
print(f"{'='*60}")
print(comparison_df.to_string(index=False))

# Save comparison
comparison_df.to_csv('comparison_results.csv', index=False)
print("\nResults saved to comparison_results.csv")

## 9. Visualization

In [ ]:
# Portfolio Value Comparison
fig = go.Figure()

ppo_values = [h['portfolio_value'] for h in ppo_test_history]
fig.add_trace(go.Scatter(
    y=ppo_values,
    mode='lines',
    name='PPO',
    line=dict(color='blue', width=2)
))

cvar_values = [h['portfolio_value'] for h in cvar_test_history]
fig.add_trace(go.Scatter(
    y=cvar_values,
    mode='lines',
    name='CVaR-PPO',
    line=dict(color='red', width=2)
))

bh_values = [h['portfolio_value'] for h in bh_test_history]
fig.add_trace(go.Scatter(
    y=bh_values,
    mode='lines',
    name='Buy & Hold',
    line=dict(color='green', width=2)
))

fig.update_layout(
    title='Portfolio Value Comparison (Test Set)',
    xaxis_title='Time Step',
    yaxis_title='Portfolio Value (USD)',
    hovermode='x unified',
    template='plotly_white',
    width=1000,
    height=500
)

fig.show()

In [ ]:
# Metrics Comparison Bar Charts
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Total Return', 'Sharpe Ratio', 'Max Drawdown (Lower is Better)', 'Win Rate')
)

methods = ['PPO', 'CVaR-PPO', 'Buy & Hold']
colors = ['blue', 'red', 'green']

# Total Return
fig.add_trace(
    go.Bar(x=methods, y=comparison_df['Total Return'], marker_color=colors, name='Total Return'),
    row=1, col=1
)

# Sharpe Ratio
fig.add_trace(
    go.Bar(x=methods, y=comparison_df['Sharpe Ratio'], marker_color=colors, name='Sharpe Ratio'),
    row=1, col=2
)

# Max Drawdown
fig.add_trace(
    go.Bar(x=methods, y=comparison_df['Max Drawdown'], marker_color=colors, name='Max Drawdown'),
    row=2, col=1
)

# Win Rate
fig.add_trace(
    go.Bar(x=methods, y=comparison_df['Win Rate'], marker_color=colors, name='Win Rate'),
    row=2, col=2
)

fig.update_layout(
    title_text='Performance Metrics Comparison',
    showlegend=False,
    template='plotly_white',
    width=1000,
    height=700
)

fig.show()

In [ ]:
# Training Progress
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Episode Rewards', 'Portfolio Values')
)

fig.add_trace(
    go.Scatter(y=ppo_training_history['episode_rewards'], mode='lines', 
               name='PPO', line=dict(color='blue')),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(y=cvar_training_history['episode_rewards'], mode='lines', 
               name='CVaR-PPO', line=dict(color='red')),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(y=ppo_training_history['portfolio_values'], mode='lines', 
               name='PPO', line=dict(color='blue'), showlegend=False),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(y=cvar_training_history['portfolio_values'], mode='lines', 
               name='CVaR-PPO', line=dict(color='red'), showlegend=False),
    row=1, col=2
)

fig.update_xaxes(title_text="Episode", row=1, col=1)
fig.update_xaxes(title_text="Episode", row=1, col=2)
fig.update_yaxes(title_text="Reward", row=1, col=1)
fig.update_yaxes(title_text="Portfolio Value (USD)", row=1, col=2)

fig.update_layout(
    title_text='Training Progress',
    template='plotly_white',
    width=1200,
    height=400
)

fig.show()

## 10. Conclusion

In [ ]:
# Summary Statistics
print(f"\n{'='*80}")
print("FINAL SUMMARY")
print(f"{'='*80}")

print(f"\nData: {CONFIG['SYMBOL']}")
print(f"Period: {CONFIG['START_DATE']} to {CONFIG['END_DATE']}")
print(f"Test Period: {test_data['date'].min()} to {test_data['date'].max()}")
print(f"Initial Balance: ${CONFIG['INITIAL_BALANCE']}")

print(f"\n{'-'*80}")
print("METHOD COMPARISON")
print(f"{'-'*80}")

best_return_method = comparison_df.loc[comparison_df['Total Return'].idxmax(), 'Method']
best_sharpe_method = comparison_df.loc[comparison_df['Sharpe Ratio'].idxmax(), 'Method']
best_drawdown_method = comparison_df.loc[comparison_df['Max Drawdown'].idxmax(), 'Method']

print(f"\nBest Total Return: {best_return_method} ({comparison_df['Total Return'].max():.4f})")
print(f"Best Sharpe Ratio: {best_sharpe_method} ({comparison_df['Sharpe Ratio'].max():.4f})")
print(f"Best Max Drawdown: {best_drawdown_method} ({comparison_df['Max Drawdown'].max():.4f})")

print(f"\n{'-'*80}")
print("OBSERVATIONS")
print(f"{'-'*80}")

print("\n1. Risk Management:")
if abs(cvar_test_metrics['max_drawdown']) < abs(ppo_test_metrics['max_drawdown']):
    print("   CVaR-PPO successfully reduces maximum drawdown compared to standard PPO")
else:
    print("   PPO shows better drawdown performance in this test period")

print("\n2. Return Performance:")
if cvar_test_metrics['total_return'] > bh_test_metrics['total_return']:
    print("   CVaR-PPO outperforms Buy & Hold strategy")
elif ppo_test_metrics['total_return'] > bh_test_metrics['total_return']:
    print("   PPO outperforms Buy & Hold strategy")
else:
    print("   Buy & Hold outperforms RL methods in this test period")

print("\n3. Sharpe Ratio:")
if cvar_test_metrics['sharpe_ratio'] > ppo_test_metrics['sharpe_ratio']:
    print("   CVaR-PPO achieves better risk-adjusted returns (higher Sharpe)")
else:
    print("   PPO achieves better risk-adjusted returns (higher Sharpe)")

print(f"\n{'='*80}")
print("CONCLUSIONS")
print(f"{'='*80}")

print("\nRisk-Sensitive RL (CVaR-PPO) vs Risk-Neutral RL (PPO):")
print("- CVaR-PPO explicitly manages tail risk through CVaR constraint")
print("- PPO focuses solely on expected returns without risk consideration")
print("- Choice depends on investor risk tolerance and market conditions")

print("\nFuture Improvements:")
print("1. Integrate news data and sentiment analysis")
print("2. Test on multiple assets and longer time periods")
print("3. Implement ensemble methods")
print("4. Add transaction cost optimization")
print("5. Deploy for real-time trading with risk controls")

print(f"\n{'='*80}")

In [ ]:
# Save models
print("Saving trained models...")

torch.save(ppo_agent.policy.state_dict(), 'ppo_model.pth')
torch.save(cvar_ppo_agent.policy.state_dict(), 'cvar_ppo_model.pth')

print("Models saved:")
print("  - ppo_model.pth")
print("  - cvar_ppo_model.pth")

# Save results
results = {
    'config': CONFIG,
    'comparison': comparison_df.to_dict(),
    'ppo_metrics': ppo_test_metrics,
    'cvar_metrics': cvar_test_metrics,
    'bh_metrics': bh_test_metrics,
    'test_period': {
        'start': str(test_data['date'].min()),
        'end': str(test_data['date'].max())
    }
}

with open('results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to results.json")
print("\nProject completed successfully!")
print(f"{'='*80}")